# Polars는 또 뭐임? 
- 폴라스는 2020년에 개발된 정형 데이터 처리 라이브러리입니다. 판다스보다 20배는 빠르대요. 

In [ ]:
import polars as pl # 폴라th
import pandas as pd

## CSV 리딩

In [ ]:
# 읽기 
df_csv = pl.read_csv("data/Fluoxetine.csv", separator=";", null_values=["", '""', 'None']) # ChEMBL은 구분자가 세미콜론임다 
df_csv2 = pd.read_csv("data/Fluoxetine.csv", sep=";")

In [ ]:
df_csv

In [ ]:
df_csv2

1. Polars는 sep=""이 아니고 separator=""임. (csv 구분자 옵션)
2. ChEMBL 데이터는 뭐가 공백인지 안 알려주면 폴라스가 음 이건 빈 문자열인가 하고 지 알아서 채워버리는 기염을 토합니다. (결측값 세다가 당황행)
2. Pandas는 21밀리초가 걸렸는데 Polars는 2밀리초 걸림
3. Polars는 열 때 shape가 같이 나옴
4. 근데 그 위에 str은 뭐니...? 

## CSV write

In [ ]:
# 여기 간단한 딕셔너리가 있습니다. 
dict = {
    '이름':['이상해씨','파이리','꼬부기','피카츄'],
    '1타입':['풀','불꽃','물','전기'],
    '2타입':['독','','','']
}

In [ ]:
# 폴라스
df = pl.DataFrame(dict)

In [ ]:
# 판다스
df2 = pd.DataFrame(dict)

- 생성하는건 그렇게 차이 안 났는데 폴라스가 1밀리초 빨랐음. 

In [ ]:
df.write_csv('test.csv') # Polas

In [ ]:
df2.to_csv('test2.csv') # Pandas

1. 파일로 쓰는건 판다스가 미묘하게 빠름
2. 대신에 Polas는 인덱스 저장이 안되기때문에 옵션을 하나 빼먹더라도 Unnamed:0같은 칼럼이 안 나옴 (판다스는 index=False 줘야 함)

## 결측값 멱살잡기

In [ ]:
df2.isna() # Pandas

In [ ]:
df.null_count() # Polas

- 아... 다 공백이라 안쳐주나봐...

In [ ]:
df_csv2.isna() # Pandas

In [ ]:
is_null_series = df_csv.select(
    pl.col("Synonyms").is_null(),
)
print(is_null_series) # Polas

- null_count() 쓰면 isna.sum 비슷한거 나옵니다. 

## 거 쿼리 있습니까

In [ ]:
# 1. 컬럼명 변경 (성공적)
df_csv = df_csv.rename({"ChEMBL ID": "ChEMBL_ID"})

# 2. 컨텍스트 및 테이블 등록
ctx = pl.SQLContext()
ctx.register("chembl_table", df_csv)

# 3. 쿼리 실행 (쌍따옴표 제거!)
result = ctx.execute("""
    SELECT * FROM chembl_table 
    WHERE ChEMBL_ID = 'CHEMBL153036'
""").collect()

result

- 아니 저기요 갑자기 SQL 나오는건 너무 정직한거 아닙니까 

In [ ]:
df_csv.filter(pl.col("ChEMBL ID") == "CHEMBL153036")

- 응 굳이 안써도됨~ 

In [ ]:
df_csv.filter(pl.col("Max Phase") > 3) # 이런것도 됩니다 

- 오. 이런것도 되는구나. 

In [ ]:
# 판다스 다들 아시죠?
df_csv2[df_csv2['ChEMBL ID'] == 'CHEMBL153036']

In [ ]:
df_csv2.query('`ChEMBL ID` == "CHEMBL153036"')

- 물론 나는 위보다 아래를 더 많이 씀. 

## .describe()나 .info같은 거 있음? 
- 다들 아시니까 판다스는 생략

In [ ]:
df_csv.schema

In [ ]:
df_csv.describe()

In [ ]:
df_csv.glimpse()

- SQL 하셨던 분들은 저기 널뜬거 보시고 오... 하실수도 있는 건인데, SQL에서 널 끼면 모든 연산이 널이 됩니다. 원래 걔는 그런 애예요. 널은 응 아니 몰라에서 몰라를 맡고 있습니다. 그냥 뭔지 몰라요 쟤는. 설문조사로 치자면 무응답이에요. 
- 어... 근데 판다스 인포에서는 알아서 칼럼 개수 이런거 세주던데... 하셨죠? 폴라스에서는 .glimpse() 쓰십쇼. 
- 근데 폴라스에는 슬픈 전설이 있어요. 씨본 쓸라면 변환해야됨... 